<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  
  <b>Module 1.1</b>  
  <span style="color:#9ca3af;">Introduction to RAG (Retrieval-Augmented Generation)</span>
</div>


## What is RAG?

**Retrieval-Augmented Generation (RAG)** is an AI framework that improves the quality of LLM-generated responses by grounding the model on external sources of knowledge. Instead of relying solely on the information the LLM was trained on, RAG fetches relevant documents at inference time and injects them directly into the prompt.

### Problems RAG Solves
| Problem | How RAG Helps |
|---|---|
| **Hallucination** | Grounds answers in retrieved facts. If the answer isn't in the context, the model can say "I don't know". |
| **Knowledge Cutoff** | Retrieves up-to-date external documents, bypassing the model's training date cutoff. |
| **Domain-specific Knowledge** | Allows ingestion of proprietary, private, or niche company data. |
| **Source Attribution** | Every answer is traceable back to a specific document chunk. |

## 1.1.2 — Architecture & Core Components

```
┌─────────────┐    embed     ┌─────────────────┐
│  User Query │────────────▶│  Vector Store   │
└─────────────┘              │  (Chroma/FAISS) │
                             └────────┬────────┘
                                      │ top-k chunks
                             ┌────────▼────────┐
                             │  Prompt Builder │
                             └────────┬────────┘
                                      │ filled prompt
                             ┌────────▼────────┐
                             │  LLM (Llama 3)  │
                             └────────┬────────┘
                                      │ answer
                             ┌────────▼────────┐
                             │  Final Response │
                             └─────────────────┘
```

### 5 Stages of RAG:
1. **Loading:** Fetching data from various sources (PDFs, Web, Databases).
2. **Splitting/Chunking:** Breaking large documents into smaller, semantically meaningful chunks.
3. **Embedding:** Converting chunks into high-dimensional vectors (arrays of numbers).
4. **Retrieval:** Taking the user query, embedding it, and finding the closest matching document vectors via cosine similarity.
5. **Generation:** Passing the retrieved context + user query to the LLM to formulate an answer.

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install langchain langchain-groq langchain-huggingface langchain-community chromadb python-dotenv

import os
from dotenv import load_dotenv
load_dotenv()   # expects GROQ_API_KEY in a .env file

## 1. Document Loading and Splitting
Instead of hardcoding documents, let's load text from a file and split it into chunks. This prevents exceeding the LLM's context window and makes retrieval more precise.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load the document (Make sure data/intro.txt exists)
loader = TextLoader("../data/intro.txt")
raw_docs = loader.load()

# 2. Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,      # Max size of chunk
    chunk_overlap=20,    # Overlap between chunks to maintain context
    length_function=len,
)
docs = text_splitter.split_documents(raw_docs)

print(f"Loaded {len(raw_docs)} document(s).")
print(f"Split into {len(docs)} chunk(s).\n")
for i, doc in enumerate(docs[:3]):
    print(f"Chunk {i+1}: {doc.page_content}")

## 2. Embeddings and Vector Store
Embeddings map text to vectors. Similar meanings are close together in vector space. We store these vectors in a Vector Database (ChromaDB) for fast retrieval.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Initialize the embedding model (Free open-source model)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Ingest chunks into ChromaDB
vectorstore = Chroma.from_documents(docs, embeddings, collection_name="rag_intro")

# 3. Create a Retriever interface (fetch top 2 most similar chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("✅ Vector store created and retriever initialized.")

## 3. The Prompt Template & LLM
We instruct the LLM to **only** use the provided context. This is crucial for preventing hallucinations.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# 1. Define the Prompt
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer ONLY using the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}
""")

# 2. Initialize the LLM (Free via Groq API)
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
print("✅ Prompt and LLM ready.")

## 4. Building the Chain (LCEL)
LangChain Expression Language (LCEL) allows us to pipe components together intuitively: `retriever -> prompt -> llm -> parser`.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(f"[Source: {d.metadata.get('source','?')}]\n{d.page_content}" for d in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("✅ LCEL Chain assembled.")

## 5. Execution and Analysis
Let's ask a question and see both the final answer AND the chunks that were retrieved.

In [ ]:
query  = "Why is RAG useful for reducing hallucinations?"

# Get the final generation
answer = chain.invoke(query)

print("Question:", query)
print("\nAnswer:")
print(answer)

print("\n" + "="*50 + "\n")

# Inspect the retrieved documents under the hood
retrieved = retriever.invoke(query)
print(f"🔍 Retrieved {len(retrieved)} chunks for context:\n")
for i, doc in enumerate(retrieved, 1):
    print(f"  [{i}] {doc.page_content}")
    print(f"       Source: {doc.metadata}\n")